# ViT image-level baseline (classification-only, NO U-Net decoder)
Scope: pretrained ViT-B/16 as **image-level damage-presence classifier** on the SAME pair split as `compare_cnn_vgg16_baselines.ipynb`.
Labels are derived from existing masks (`y=1{damage_ratio>=0.01}`); NO new annotation, NO Dice/IoU for this head, NO ranking claim vs segmentation models.
Requires `vit-keras==0.1.0` + `tensorflow-addons==0.19.0` + `setuptools<81` (`pip install "vit-keras==0.1.0" "tensorflow-addons==0.19.0" "setuptools<81"`). Kombinasi ini yang cocok dgn TF 2.10/Keras 2.10: vit-keras 0.2.x butuh Keras 3, tensorflow-addons lain bentrok dgn TF 2.10, setuptools>=81 menghapus pkg_resources. keras-hub TIDAK dipakai: butuh Keras 3, merusak env ini.
Run from `backend/model/`. Writes only to `comparison_cls/`. Existing checkpoints/results untouched.


In [1]:
import os, glob, json
import numpy as np
import cv2
import tensorflow as tf
for _g in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(_g, True)
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix)
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED)
IMG_SIZE = (256, 256)
BATCH = 8
EPOCHS = 15
LR = 1e-4
FINE_LR = 1e-5
TAU_LABEL = 0.01  # image positive if >=1% damage pixels (matches Low/Medium break)
TRAINING_DIR = os.path.join('..', '..', 'training')
IMAGES_DIR = os.path.join(TRAINING_DIR, 'images')
TARGETS_DIR = os.path.join(TRAINING_DIR, 'targets')
OUTPUT_DIR = 'comparison_cls'
os.makedirs(OUTPUT_DIR, exist_ok=True)
_tf_version = tuple(int(p) for p in tf.__version__.split('.')[:2])
# TF lawas tidak punya kernel CUDA native utk compute capability >= 12 (mis. RTX 50 series);
# tanpa guard ini Epoch 1 hang saat PTX JIT compile. Pola sama dgn compare_cnn_vgg16_baselines.ipynb.
for _g in tf.config.list_physical_devices('GPU'):
    _det = tf.config.experimental.get_device_details(_g)
    _cap = _det.get('compute_capability')
    if _cap and _cap[0] >= 12 and _tf_version <= (2, 10):
        tf.config.set_visible_devices([], 'GPU')
        print('GPU disembunyikan (CUDA incompatible, compute capability', _cap, ') -> lanjut CPU.')
        break
print('TF:', tf.__version__, '| GPU terlihat:', tf.config.list_physical_devices('GPU'))

c:\Users\Jase.LAPTOP-UM736EL9\AppData\Local\Programs\Python\Python310\lib\site-packages\google\api_core\_python_version_support.py:242: FutureWarning: You are using a non-supported Python version (3.10.0). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.11, and then update google.api_core.
  warnings.warn(message, FutureWarning)


GPU disembunyikan (CUDA incompatible, compute capability (12, 0) ) -> lanjut CPU.
TF: 2.10.0 | GPU terlihat: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
# SAME pair discovery + SAME split protocol as compare_cnn_vgg16_baselines.ipynb
post_images = sorted(glob.glob(os.path.join(IMAGES_DIR, '*_post_disaster.png')))
image_pairs, target_paths = [], []
for post_path in post_images:
    pre_path = post_path.replace('_post_disaster.png', '_pre_disaster.png')
    scene_id = os.path.basename(post_path).replace('_post_disaster.png', '')
    mask_path = os.path.join(TARGETS_DIR, f'{scene_id}_post_disaster_target.png')
    if not os.path.exists(mask_path): continue
    if not os.path.exists(pre_path): pre_path = post_path
    image_pairs.append((pre_path, post_path)); target_paths.append(mask_path)
tr_p, te_p, tr_m, te_m = train_test_split(image_pairs, target_paths, test_size=0.30, random_state=SEED)
va_p, ts_p, va_m, ts_m = train_test_split(te_p, te_m, test_size=0.50, random_state=SEED)
print(f'Total {len(image_pairs)} | train {len(tr_p)} | val {len(va_p)} | test {len(ts_p)}')

Total 2799 | train 1959 | val 420 | test 420


In [3]:
# Image-level pipeline: POST RGB only (ImageNet stems expect 3ch) + label from mask ratio
def load_cls(post_path, mask_path):
    pp = os.fsdecode(post_path.numpy() if hasattr(post_path, 'numpy') else post_path)
    mp = os.fsdecode(mask_path.numpy() if hasattr(mask_path, 'numpy') else mask_path)
    img = cv2.imread(pp)
    if img is None: raise ValueError(f'Gagal membaca citra: {pp}')
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, IMG_SIZE).astype(np.float32) / 255.0
    m = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
    if m is None: raise ValueError(f'Gagal membaca mask: {mp}')
    m = cv2.resize(m, IMG_SIZE, interpolation=cv2.INTER_NEAREST)
    return img, np.float32((m > 0).mean() >= TAU_LABEL)
def tf_load(pp, mp):
    img, lab = tf.py_function(load_cls, [pp, mp], [tf.float32, tf.float32])
    img.set_shape([*IMG_SIZE, 3]); lab.set_shape([])
    return img, lab
def aug(img, lab):
    img = tf.cond(tf.random.uniform(()) > 0.5, lambda: tf.image.flip_left_right(img), lambda: img)
    img = tf.cond(tf.random.uniform(()) > 0.5, lambda: tf.image.flip_up_down(img), lambda: img)
    return img, lab
def make_ds(pairs, masks, training=False):
    post = [p[1] for p in pairs]
    ds = tf.data.Dataset.from_tensor_slices((post, masks))
    if training: ds = ds.shuffle(len(post), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(tf_load, num_parallel_calls=tf.data.AUTOTUNE)
    if training: ds = ds.map(aug, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH).prefetch(tf.data.AUTOTUNE)
train_ds, val_ds, test_ds = make_ds(tr_p, tr_m, True), make_ds(va_p, va_m), make_ds(ts_p, ts_m)
print('splits ready:', len(tr_p), len(va_p), len(ts_p))

splits ready: 1959 420 420


In [4]:
# SMOKE TEST (wajib sebelum fit): pastikan 1 batch mengalir + ukur waktunya.
import time as _t
_t0 = _t.time()
for _xb, _yb in train_ds.take(1):
    print('batch:', tuple(_xb.shape), tuple(_yb.shape),
          '| fraksi positif:', round(float(_yb.numpy().mean()), 4),
          f'| {_t.time()-_t0:.1f}s')
print('OK - pipeline mengalir, aman lanjut ke build/train.')
print('Jika CELL INI yang hang >5 mnt: masalahnya I/O data (path/corrupt), bukan GPU.')

batch: (8, 256, 256, 3) (8,) | fraksi positif: 0.375 | 0.1s
OK - pipeline mengalir, aman lanjut ke build/train.
Jika CELL INI yang hang >5 mnt: masalahnya I/O data (path/corrupt), bukan GPU.


In [5]:
def build_vit():
    # vit-keras (faustomorales): ViT-B/16 ImageNet-pretrained utk tf.keras era Keras-2.
    # include_top=False -> output CLS token (batch, 768); head klasifikasi di bawah.
    try:
        from vit_keras import vit as _vit
    except ImportError:
        raise RuntimeError("vit-keras not installed. Run: pip install \"vit-keras==0.1.0\" \"tensorflow-addons==0.19.0\" \"setuptools<81\"")
    backbone = _vit.vit_b16(image_size=224, pretrained=True,
                            include_top=False, pretrained_top=False)
    backbone.trainable = False
    inp = Input((*IMG_SIZE, 3))
    x = layers.Resizing(224, 224)(inp)  # ViT-B/16 native resolution
    x = layers.Rescaling(scale=255.0)(x)  # [0,1] -> [0,255]
    x = layers.Rescaling(scale=1/127.5, offset=-1.0)(x)  # == vit.preprocess_inputs, tf-mode [-1,1]
    feats = backbone(x)
    x = layers.Dropout(0.2)(feats)
    out = layers.Dense(1, activation='sigmoid')(x)
    return Model(inp, out, name='ViT_B16_cls')

In [6]:
def collect(model, ds):
    yt, yp = [], []
    for xb, yb in ds:
        yt.append(yb.numpy()); yp.append(model.predict(xb, verbose=0).ravel())
    return np.concatenate(yt), np.concatenate(yp)
def pick_threshold(model, ds, grid=np.arange(0.10, 0.91, 0.05)):
    yt, yp = collect(model, ds)
    best_t, best_f1 = 0.5, -1.0
    for t in grid:
        f1 = f1_score(yt, yp >= t, zero_division=0)
        if f1 > best_f1: best_t, best_f1 = float(t), float(f1)
    return best_t
def eval_cls(model, ds, thr):
    yt, yp = collect(model, ds)
    pred = yp >= thr
    return {'threshold': thr, 'Accuracy': float(accuracy_score(yt, pred)),
        'Precision': float(precision_score(yt, pred, zero_division=0)),
        'Recall': float(recall_score(yt, pred, zero_division=0)),
        'F1': float(f1_score(yt, pred, zero_division=0)),
        'ROC_AUC': float(roc_auc_score(yt, yp)), 'PR_AUC': float(average_precision_score(yt, yp)),
        'TN_FP_FN_TP': [int(v) for v in confusion_matrix(yt, pred).ravel()]}
def train_eval(build_fn, tag, lr=LR, fine_tune=False, ft_lr=FINE_LR):
    m = build_fn() if not fine_tune else build_fn(fine_tune=True)
    m.compile(optimizer=Adam(learning_rate=(ft_lr if fine_tune else lr)),
              loss='binary_crossentropy', metrics=['accuracy'])
    ckpt = os.path.join(OUTPUT_DIR, f'{tag}_best.keras')
    m.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=[
        ModelCheckpoint(ckpt, monitor='val_loss', mode='min', save_best_only=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1),
        EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)])
    m.save(os.path.join(OUTPUT_DIR, f'{tag}_final.keras'))
    thr = pick_threshold(m, val_ds)
    row = eval_cls(m, test_ds, thr)
    row['model'] = tag
    return m, row

In [7]:
import pandas as pd
results = []
vit_model, vit_row = train_eval(build_vit, 'vit_b16_cls')
results.append(vit_row)
comp = pd.DataFrame(results).set_index('model')
comp.to_csv(os.path.join(OUTPUT_DIR, 'cls_vit_results.csv'))
json.dump(results, open(os.path.join(OUTPUT_DIR, 'cls_vit_results.json'), 'w'), indent=2)
comp  # Accuracy/Precision/Recall/F1/ROC-AUC/PR-AUC at val-selected threshold


c:\Users\Jase.LAPTOP-UM736EL9\AppData\Local\Programs\Python\Python310\lib\site-packages\vit_keras\utils.py:7: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
c:\Users\Jase.LAPTOP-UM736EL9\AppData\Local\Programs\Python\Python310\lib\site-packages\vit_keras\utils.py:81: UserWarning: Resizing position embeddings from 24, 24 to 14, 14
  warnings.warn(


Epoch 1/15
245/245 [==============================] - ETA: 0s - loss: 0.6962 - accuracy: 0.6212
Epoch 1: val_loss improved from inf to 0.48962, saving model to comparison_cls\vit_b16_cls_best.keras
245/245 [==============================] - 299s 1s/step - loss: 0.6962 - accuracy: 0.6212 - val_loss: 0.4896 - val_accuracy: 0.7690 - lr: 1.0000e-04
Epoch 2/15
245/245 [==============================] - ETA: 0s - loss: 0.5030 - accuracy: 0.7606
Epoch 2: val_loss improved from 0.48962 to 0.38485, saving model to comparison_cls\vit_b16_cls_best.keras
245/245 [==============================] - 360s 1s/step - loss: 0.5030 - accuracy: 0.7606 - val_loss: 0.3849 - val_accuracy: 0.8405 - lr: 1.0000e-04
Epoch 3/15
245/245 [==============================] - ETA: 0s - loss: 0.4365 - accuracy: 0.8086
Epoch 3: val_loss improved from 0.38485 to 0.34232, saving model to comparison_cls\vit_b16_cls_best.keras
245/245 [==============================] - 330s 1s/step - loss: 0.4365 - accuracy: 0.8086 - val_loss

,threshold,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC,TN_FP_FN_TP
model,,,,,,,,
vit_b16_cls,0.55,0.888095,0.906667,0.886957,0.896703,0.961991,0.971112,"[169, 21, 26, 204]"
